# Desafio 3 - ZettaLab

<img src="https://ufla.br/images/noticias/2020/04_abr/logo_zetta.png">

## Descrição do Problema

## Descrição da Solução

## Equipe

- Luciana Laibe Santos Silva, Comunicação e Marketing (9° Período, Graduação em Ciências Biológicas)
- Estevão Augusto da Fonseca Santos, Ciência e Governança de Dados (6° Período, Graduação em Ciência de Computação)
- Hugo Dias Pontello, Desenvolvimento de Software (5° Período, Graduação em Sistemas de Informação)
- Lorrana Verdi Flores, Desenvolvimento de Software (6° Período, Doutorado em Biotecnologia Vegetal)
- Bruna Oliveira Pereira, Geotecnologia (4° Período, Graduação em Engenharia Florestal)
- Geovanna Alexandre Possidonio, Gestão de Projetos (4° Período, Graduação em Administração)

## Importação de Bibliotecas

In [1]:
import pandas as pd             # Biblioteca para manipulação e análise de dados
import numpy as np              # Biblioteca para calculo de vetores e matrizes
import sys                      # Biblioteca para acessar variaveis e funções que interagem fortemente com o interpretador
import os                       # Biblioteca para interação com arquivos e diretórios a nivel sistema operacional
import basedosdados as bd       # Biblioteca para acessar o datalake público do site BasedosDados
import glob                     # Biblioteca para processamento em lote
from pathlib import Path        # Biblioteca para a manipulação de caminhos do sistema a nivel orientado a objetos
import gzip                     # Biblioteca para compressão e decompressão de dados usando o formato gzip
import shutil                   # Biblioteca para manipulação de arquivos e diretorios
import re

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import config_path          # Módulo que salva todos os caminhos de diretórios utilizados no projeto

from scripts import features        # Módulo de criação e manipulação de features
from scripts import modeling        # Módulo de modelagem de IA
from scripts import utils           # Módulo de utilitários genéricos
from scripts import pre_processing  # Módulo de pré-processamento e obtenção de dados.

from dotenv import load_dotenv      # Biblioteca para carregar variáveis de ambiente de arquivos .env
load_dotenv()
GOOGLE_CLOUD_ID_PROJECT = os.getenv("GOOGLE_CLOUD_ID_PROJECT") # Coloca o ID do projeto do Google Cloud numa constante

## Obtenção e Tratamento de Dados

#### [Código de Múnicipios IBGE](https://www.ibge.gov.br/explica/codigos-dos-municipios.php)

A Tabela de Códigos de Municípios do IBGE apresenta a lista dos municípios brasileiros associados a um código composto de 7 dígitos, sendo os dois primeiros referentes ao código da Unidade da Federação.

É atualizada sistematicamente de forma a incluir as alterações decorrentes do desdobramento de municípios e, conseqüentemente, da criação de novos municípios, mudanças de nome dos municípios, como também de processos de fusão que resultam na extinção ou modificação de nome de algum município. 

In [2]:
url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/estrutura_territorial/divisao_territorial/2024/DTB_2024.zip"

if not os.path.exists(config_path.RAW_DATA_DIRECTORY_PATH / "RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods"):
    zip_file_path = utils.download_file(url, "codigo_municipios_ibge.zip", config_path.RAW_DATA_DIRECTORY_PATH)
    utils.unzip_and_clean(zip_file_path, ['RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods'], config_path.RAW_DATA_DIRECTORY_PATH)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/codigo_municipios_ibge.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\Distritos_novos_e_extintos.ods
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\Distritos_novos_e_extintos.xls
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_DISTRITOS.ods
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_DISTRITOS.xls
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.xls
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_SUBDISTRITOS.ods
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_SUBDISTRITOS.xls
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\codigo_municipios_ibge.zip
Arquivos mantidos: [WindowsPath(

In [3]:
df = pd.read_excel(f"{config_path.RAW_DATA_DIRECTORY_PATH}/RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods")
df.drop('Unnamed: 9', axis=1, inplace=True)
df.columns = df.loc[5].to_list()
df.drop(index=5, inplace=True)
df.reset_index(drop=True, inplace=True)
df.rename({'UF' : 'ID_UF',
           'Código Município Completo' : 'ID_Município'
           }, axis=1, inplace=True)
df.dropna(axis=0, inplace=True) # Linhas
df.reset_index(inplace=True)
df.drop(columns=['index'], inplace=True)

In [4]:
df

,ID_UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,ID_Município,Nome_Município
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,00015,1100015,Alta Floresta D'Oeste
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,00379,1100379,Alto Alegre dos Parecis
2,11,Rondônia,1101,Porto Velho,110002,Ariquemes,00403,1100403,Alto Paraíso
3,11,Rondônia,1102,Ji-Paraná,110004,Ji-Paraná,00346,1100346,Alvorada D'Oeste
4,11,Rondônia,1101,Porto Velho,110002,Ariquemes,00023,1100023,Ariquemes
...,...,...,...,...,...,...,...,...,...
5566,52,Goiás,5201,Goiânia,520002,Anápolis,22005,5222005,Vianópolis
5567,52,Goiás,5202,Itumbiara,520009,Piracanjuba,22054,5222054,Vicentinópolis
5568,52,Goiás,5206,Luziânia - Águas Lindas de Goiás,520022,Flores de Goiás,22203,5222203,Vila Boa
5569,52,Goiás,5205,Porangatu - Uruaçu,520018,Ceres - Rialma - Goianésia,22302,5222302,Vila Propício


In [5]:
df.drop(columns=[
    "Região Geográfica Intermediária", 
    "Nome Região Geográfica Intermediária",
    "Região Geográfica Imediata",
    "Nome Região Geográfica Imediata",
    "Município"
    ], axis=1, inplace=True)

df_uf = df[["ID_UF", "Nome_UF"]].drop_duplicates()
df_uf.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/codigos_ufs.csv", index=False)
df_mun = df[["ID_UF", "Nome_UF", "ID_Município", "Nome_Município"]]
df_mun.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/codigos_municipios.csv", index=False)

In [6]:
df_uf

,ID_UF,Nome_UF
0,11,Rondônia
52,12,Acre
74,13,Amazonas
136,14,Roraima
151,15,Pará
295,16,Amapá
311,17,Tocantins
450,21,Maranhão
667,22,Piauí
891,23,Ceará


In [7]:
df_mun

,ID_UF,Nome_UF,ID_Município,Nome_Município
0,11,Rondônia,1100015,Alta Floresta D'Oeste
1,11,Rondônia,1100379,Alto Alegre dos Parecis
2,11,Rondônia,1100403,Alto Paraíso
3,11,Rondônia,1100346,Alvorada D'Oeste
4,11,Rondônia,1100023,Ariquemes
...,...,...,...,...
5566,52,Goiás,5222005,Vianópolis
5567,52,Goiás,5222054,Vicentinópolis
5568,52,Goiás,5222203,Vila Boa
5569,52,Goiás,5222302,Vila Propício


In [8]:
del df_uf
del df_mun

#### [Areas Territoriais - IBGE](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/estrutura-territorial/15761-areas-dos-municipios.html?t=acesso-ao-produto&c=1)

O redimensionamento dos valores de áreas é próprio da evolução das geotecnologias aplicadas no monitoramento da dinâmica da divisão territorial brasileira, que implica na atualização periódica dos valores das áreas estaduais e municipais com a utilização continuada de melhores técnicas e de melhores insumos de produção, além de refletir as eventuais alterações nos limites político-administrativos por justificativas legais ou judiciais.

In [9]:
url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/estrutura_territorial/areas_territoriais/2024/AR_BR_RG_UF_RGINT_RGI_MUN_2024.ods"

if not os.path.exists(config_path.RAW_DATA_DIRECTORY_PATH / "area_territorial_brasil_regioes_ufs_muns.ods"):
    file_name = utils.download_file(url, "area_territorial_brasil_regioes_ufs_muns.ods", config_path.RAW_DATA_DIRECTORY_PATH)
else:
    file_name = config_path.RAW_DATA_DIRECTORY_PATH / "area_territorial_brasil_regioes_ufs_muns.ods"
df_mun = pd.read_excel(file_name, "AR_BR_MUN_2024")
df_uf = pd.read_excel(file_name, "AR_BR_UF_2024")
df_rg = pd.read_excel(file_name, "AR_BR_RG_2024")

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/area_territorial_brasil_regioes_ufs_muns.ods


In [10]:
df_uf

,CD_UF,NM_UF,NM_UF_SIGLA,AR_UF_2024
0,11,Rondônia,RO,237754.171
1,12,Acre,AC,164082.960
2,13,Amazonas,AM,1558706.127
3,14,Roraima,RR,223505.385
4,15,Pará,PA,1245828.829
5,16,Amapá,AP,142253.880
6,17,Tocantins,TO,277423.627
7,21,Maranhão,MA,329651.478
8,22,Piauí,PI,251755.499
9,23,Ceará,CE,148894.444


In [11]:
df_rg

,CD_RG,NM_RG,NM_RG_SIGLA,AR_RG_2024
0,1,Norte,N,3849554.979
1,2,Nordeste,NE,1552175.419
2,3,Sudeste,SE,924558.342
3,4,Sul,S,576736.821
4,5,Centro-oeste,CO,1606354.015
5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN
7,OBS: Estes valores de área foram calculados a ...,NaN,NaN,NaN


In [12]:
df_mun

,CD_UF,NM_UF,NM_UF_SIGLA,CD_MUN,NM_MUN,AR_MUN_2024
0,11,Rondônia,RO,1101203.0,Ministro Andreazza,798.083
1,11,Rondônia,RO,1101708.0,Urupá,831.857
2,11,Rondônia,RO,1100403.0,Alto Paraíso,2651.991
3,11,Rondônia,RO,1100452.0,Buritis,3265.810
4,11,Rondônia,RO,1101757.0,Vale do Anari,3135.106
...,...,...,...,...,...,...
5570,52,Goiás,GO,5213855.0,Morro Agudo de Goiás,282.333
5571,52,Goiás,GO,5201207.0,Anhanguera,55.569
5572,53,Distrito Federal,DF,5300108.0,Brasília,5760.783
5573,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_mun.dropna(how='any', axis=0, inplace=True)
df_mun.rename(columns={ 'CD_UF' : 'ID_UF',
                'NM_UF' : 'Nome_UF',
                'NM_UF_SIGLA' : 'UF_SIGLA',
                'CD_MUN' : 'ID_Município',
                'AR_MUN_2024' : 'AreaKm2_Mun_2024',
                'NM_MUN' : 'Nome_Municipio'
                }, inplace=True)
df_mun.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/municipio_area_2024.csv", index=False)
df_mun

,ID_UF,Nome_UF,UF_SIGLA,ID_Município,Nome_Municipio,AreaKm2_Mun_2024
0,11,Rondônia,RO,1101203.0,Ministro Andreazza,798.083
1,11,Rondônia,RO,1101708.0,Urupá,831.857
2,11,Rondônia,RO,1100403.0,Alto Paraíso,2651.991
3,11,Rondônia,RO,1100452.0,Buritis,3265.810
4,11,Rondônia,RO,1101757.0,Vale do Anari,3135.106
...,...,...,...,...,...,...
5568,52,Goiás,GO,5202502.0,Aruanã,3054.773
5569,52,Goiás,GO,5210802.0,Itajá,2082.737
5570,52,Goiás,GO,5213855.0,Morro Agudo de Goiás,282.333
5571,52,Goiás,GO,5201207.0,Anhanguera,55.569


In [14]:
df_uf.dropna(how='any', axis=0, inplace=True)
df_uf.rename(columns={ 'CD_UF' : 'ID_UF',
               'NM_UF' : 'Nome_UF',
               'NM_UF_SIGLA' : 'UF_SIGLA',
               'AR_UF_2024' : 'AreaKm2_UF_2024'   
            }, inplace=True)
df_uf.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/uf_area_2024.csv", index=False)
df_uf

,ID_UF,Nome_UF,UF_SIGLA,AreaKm2_UF_2024
0,11,Rondônia,RO,237754.171
1,12,Acre,AC,164082.960
2,13,Amazonas,AM,1558706.127
3,14,Roraima,RR,223505.385
4,15,Pará,PA,1245828.829
5,16,Amapá,AP,142253.880
6,17,Tocantins,TO,277423.627
7,21,Maranhão,MA,329651.478
8,22,Piauí,PI,251755.499
9,23,Ceará,CE,148894.444


In [15]:
df_rg.dropna(how='any', axis=0, inplace=True)
df_rg.rename(columns={'CD_RG' : 'ID_RG',
             'NM_RG' : 'Nome_RG',
             'NM_RG_SIGLA' : 'RG_SIGLA',
             'AR_RG_2024' : 'AreaKm2_RG_2024'}, 
             inplace=True)
df_rg.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/rg_area_2024.csv", index=False)
df_rg

,ID_RG,Nome_RG,RG_SIGLA,AreaKm2_RG_2024
0,1,Norte,N,3849554.979
1,2,Nordeste,NE,1552175.419
2,3,Sudeste,SE,924558.342
3,4,Sul,S,576736.821
4,5,Centro-oeste,CO,1606354.015


#### BDQUEIMADAS - Dados

O BDQueimadas é o banco de dados de queimadas mantido pelo Instituto Nacional de Pesquisas Espaciais (INPE), que disponibiliza informações sobre focos de calor detectados por satélites. Esses focos indicam possíveis ocorrências de queimadas ou incêndios na vegetação, sendo monitorados continuamente em todo o território brasileiro e em outras regiões da América Latina.

Os dados fornecidos incluem informações como localização geográfica (latitude e longitude), data e hora da detecção, satélite responsável, além de atributos complementares como bioma, estado e intensidade do foco. Essas informações são coletadas por meio de sensoriamento remoto e organizadas em uma plataforma acessível ao público.

O BDQueimadas é amplamente utilizado para monitoramento ambiental, apoio à tomada de decisão por órgãos públicos, estudos científicos e análises de dados. A plataforma permite acesso aos dados por meio de downloads em formatos como CSV e GeoJSON, além de visualizações interativas e serviços que possibilitam integração com aplicações externas.

##### [BDQUEIMADAS - Monitoramento dos Focos Ativos por Países](https://terrabrasilis.dpi.inpe.br/queimadas/situacao-atual/estatisticas/estatisticas_paises/)

In [16]:
url = "https://terrabrasilis.dpi.inpe.br/queimadas/situacao-atual/media/csv_estatisticas/historico_pais_brasil.csv"

csv_file_path = utils.download_file(url, "comparacao_foco_ativos_brasil.csv")
df = pd.read_csv(csv_file_path)
df

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/comparacao_foco_ativos_brasil.csv


,Unnamed: 0,Janeiro,Fevereiro,Março,Abril,Maio,Junho,Julho,Agosto,Setembro,Outubro,Novembro,Dezembro,Total
0,1998,NaN,NaN,NaN,NaN,NaN,3551.0,8067.0,35551.0,41976.0,23499.0,6804.0,4448.0,123896.0
1,1999,1081.0,1284.0,667.0,717.0,1811.0,3632.0,8758.0,39492.0,36914.0,27017.0,8863.0,4376.0,134612.0
2,2000,778.0,562.0,848.0,538.0,2097.0,6274.0,4740.0,22204.0,23293.0,27332.0,8399.0,4465.0,101530.0
3,2001,547.0,1060.0,1267.0,1081.0,2090.0,8405.0,6488.0,31838.0,39829.0,31039.0,15640.0,6200.0,145484.0
4,2002,1653.0,1569.0,1678.0,1683.0,3816.0,10845.0,18080.0,72412.0,93417.0,59257.0,39913.0,17091.0,321414.0
5,2003,6697.0,3099.0,3549.0,3643.0,6448.0,16752.0,30391.0,57004.0,97758.0,57495.0,35421.0,22980.0,341237.0
6,2004,3883.0,1932.0,2928.0,2956.0,6609.0,18024.0,30356.0,64067.0,121395.0,54292.0,45364.0,28639.0,380445.0
7,2005,7057.0,2898.0,2528.0,2743.0,5075.0,7854.0,30238.0,90729.0,102455.0,65023.0,31631.0,14332.0,362563.0
8,2006,4531.0,2387.0,2426.0,2269.0,4313.0,7601.0,17788.0,54630.0,76475.0,32043.0,29302.0,15414.0,249179.0
9,2007,4220.0,2761.0,3340.0,2550.0,5123.0,12716.0,19931.0,91085.0,141220.0,67228.0,31421.0,12320.0,393915.0


In [17]:
df.rename(columns={'Unnamed: 0' : 'Ano'}, 
          inplace=True)
df.dropna(how='any', axis=0, inplace=True)
df.to_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "comparacao_foco_ativos_brasil.csv", index=False)

In [18]:
df

,Ano,Janeiro,Fevereiro,Março,Abril,Maio,Junho,Julho,Agosto,Setembro,Outubro,Novembro,Dezembro,Total
1,1999,1081.0,1284.0,667.0,717.0,1811.0,3632.0,8758.0,39492.0,36914.0,27017.0,8863.0,4376.0,134612.0
2,2000,778.0,562.0,848.0,538.0,2097.0,6274.0,4740.0,22204.0,23293.0,27332.0,8399.0,4465.0,101530.0
3,2001,547.0,1060.0,1267.0,1081.0,2090.0,8405.0,6488.0,31838.0,39829.0,31039.0,15640.0,6200.0,145484.0
4,2002,1653.0,1569.0,1678.0,1683.0,3816.0,10845.0,18080.0,72412.0,93417.0,59257.0,39913.0,17091.0,321414.0
5,2003,6697.0,3099.0,3549.0,3643.0,6448.0,16752.0,30391.0,57004.0,97758.0,57495.0,35421.0,22980.0,341237.0
6,2004,3883.0,1932.0,2928.0,2956.0,6609.0,18024.0,30356.0,64067.0,121395.0,54292.0,45364.0,28639.0,380445.0
7,2005,7057.0,2898.0,2528.0,2743.0,5075.0,7854.0,30238.0,90729.0,102455.0,65023.0,31631.0,14332.0,362563.0
8,2006,4531.0,2387.0,2426.0,2269.0,4313.0,7601.0,17788.0,54630.0,76475.0,32043.0,29302.0,15414.0,249179.0
9,2007,4220.0,2761.0,3340.0,2550.0,5123.0,12716.0,19931.0,91085.0,141220.0,67228.0,31421.0,12320.0,393915.0
10,2008,2777.0,1751.0,1887.0,1906.0,2951.0,4594.0,14029.0,34431.0,50671.0,51784.0,30724.0,14428.0,211933.0


##### [BDQUEIMADAS - CSV - 2025 a 2022](https://terrabrasilis.dpi.inpe.br/queimadas/bdqueimadas/#exportar-dados)

In [19]:
def limpar_nome_arquivo(nome):
    # Adicionamos o espaço " " dentro da lista de permitidos [ ... ]
    # A lógica agora é: substitua o que NÃO for letra, número, ponto, hífen OU espaço.
    return re.sub(r'[^\w\d.\- ]', '_', nome)

urls = ["https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=ec9ed2ac-18c7-5e58-b05c-f2e5806f99eb&token=476fc698-b359-5459-b529-196ec5db488a&id=b0c07fb2-1660-5f3f-b6c9-994d0468fd2c",
       "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=b0589a58-47a5-5377-9839-5904fc14afc9&token=476fc698-b359-5459-b529-196ec5db488a&id=f712fdb9-d2b0-5ba8-80e4-937bcaf56255",
       "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=008d5526-dde7-508d-8cb3-3f30d74bf2e3&token=476fc698-b359-5459-b529-196ec5db488a&id=df75caa2-7dad-5494-ab2a-559483f1f56f",
       "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=c4a23f47-b03d-5611-bfd5-9081671d667d&token=476fc698-b359-5459-b529-196ec5db488a&id=b799b6f9-b9bc-5889-9815-ad1b2996ab1e"]

nomes_zips = ["bdqueimadas_2025-01-01_2025-12-31.zip",
              "bdqueimadas_2024-01-01_2024-12-31.zip",
              "bdqueimadas_2023-01-01_2023-12-31.zip",
              "bdqueimadas_2022-01-01_2022-12-31.zip"]

nome_arquivos = ["bdqueimadas_2025-01-01_2025-12-31.csv",
                 "bdqueimadas_2024-01-01_2024-12-31.csv",
                 "bdqueimadas_2023-01-01_2023-12-31.csv",
                 "bdqueimadas_2022-01-01_2022-12-31.csv"]

for url, nome_arquivo, nome_zip in zip(urls, nome_arquivos, nomes_zips):
    zip_file_path = utils.download_file(url, nome_zip, config_path.RAW_DATA_DIRECTORY_PATH)
    file_content = utils.unzip_and_clean(zip_file_path, 'all', config_path.RAW_DATA_DIRECTORY_PATH)
    print(file_content)
    
    origem_suja = file_content[0]
    diretorio = origem_suja.parent.absolute()
    nome_sujo = origem_suja.name
    nome_limpo = limpar_nome_arquivo(nome_sujo)
    
    caminho_corrigido = diretorio / nome_limpo
    
    
    os.rename(caminho_corrigido, config_path.RAW_DATA_DIRECTORY_PATH / nome_arquivo)
    
    df = pd.read_csv(config_path.RAW_DATA_DIRECTORY_PATH / nome_arquivo)
    df = df.loc[df['Estado'] == "MINAS GERAIS"]
    
    df.dropna(how='any', axis=0, inplace=True)
    df.reset_index(inplace=True)

    df.rename(columns={
                'Estado' : 'Nome_UF',
                'Municipio' : 'Nome_Município'
            }, inplace=True)

    df['DataHora'] = pd.to_datetime(df['DataHora'], format='%Y/%m/%d %H:%M:%S')
    
    # Extrai colunas separadas
    df['Data'] = df['DataHora'].dt.date       # Só a data (2025-01-01)
    df['Hora'] = df['DataHora'].dt.time       # Só a hora (03:49:00)
    df['Ano'] = df['DataHora'].dt.year        # Ano
    df['Mes'] = df['DataHora'].dt.month       # Mês numérico
    df['Dia'] = df['DataHora'].dt.day         # Dia
    df['Hora_decimal'] = df['DataHora'].dt.hour + df['DataHora'].dt.minute/60  # Hora em decimal

    df_uf = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_ufs.csv")
    df_uf['Nome_UF'] = df_uf['Nome_UF'].str.upper()
    df_mun = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_municipios.csv")
    df_mun = df_mun.loc[:, ['ID_Município','Nome_Município']]
    df_mun['Nome_Município'] = df['Nome_Município'].str.upper()

    df = pd.merge(left=df, right=df_uf, how="inner", on=['Nome_UF'])
    df = pd.merge(left=df, right=df_mun, how="inner", on=['Nome_Município'])
    df.drop(columns='index', inplace=True)
    
    df.to_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / (nome_arquivo.replace('.csv', '.parquet')), index=False)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/bdqueimadas_2025-01-01_2025-12-31.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
Arquivos mantidos: [WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/bdqueimadas_2025-01-01_2025-12-31.csv')]
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\bdqueimadas_2025-01-01_2025-12-31.zip
[WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/bdqueimadas_2025-01-01_2025-12-31.csv')]
Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/bdqueimadas_2024-01-01_2024-12-31.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
Arquivos mantidos: [WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/exportador_2025-09-25 20:02:36.991309.csv')]
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\bdqueimadas_2024-01-01_2024-12-31.zip
[WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-

In [20]:
df = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "bdqueimadas_2025-01-01_2025-12-31.parquet")

df

,DataHora,Satelite,Pais,Nome_UF,Nome_Município,Bioma,DiaSemChuva,Precipitacao,RiscoFogo,FRP,Latitude,Longitude,Data,Hora,Ano,Mes,Dia,Hora_decimal,ID_UF,ID_Município
0,2025-01-01 03:49:00,NOAA-20,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,1.02,-999.00,1.3,-20.76235,-46.77194,2025-01-01,03:49:00,2025,1,1,3.816667,31,1100015
1,2025-01-01 03:49:00,NOAA-20,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,1.02,-999.00,1.3,-20.76235,-46.77194,2025-01-01,03:49:00,2025,1,1,3.816667,31,1100379
2,2025-01-01 03:49:00,NOAA-20,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,1.02,-999.00,1.3,-20.76235,-46.77194,2025-01-01,03:49:00,2025,1,1,3.816667,31,1100403
3,2025-01-01 03:49:00,NOAA-20,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,1.02,-999.00,1.3,-20.76235,-46.77194,2025-01-01,03:49:00,2025,1,1,3.816667,31,1300409
4,2025-01-01 03:49:00,NOAA-20,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,1.02,-999.00,1.3,-20.76235,-46.77194,2025-01-01,03:49:00,2025,1,1,3.816667,31,1300607
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6740455,2025-12-31 20:00:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,132.2,-17.37620,-41.50410,2025-12-31,20:00:00,2025,12,31,20.000000,31,2805000
6740456,2025-12-31 20:00:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,132.2,-17.37620,-41.50410,2025-12-31,20:00:00,2025,12,31,20.000000,31,3537602
6740457,2025-12-31 20:00:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,132.2,-17.37620,-41.50410,2025-12-31,20:00:00,2025,12,31,20.000000,31,4305371
6740458,2025-12-31 20:00:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,132.2,-17.37620,-41.50410,2025-12-31,20:00:00,2025,12,31,20.000000,31,4306734


### [Banco de Dados Meteorológicos (BDMEP) - BaseDosDados](https://basedosdados.org/dataset/782c5607-9f69-4e12-b0d5-aa0f1a7a94e2?table=2c7fdc3d-f2ed-4c78-84b8-d9c792a06703)



O BDMEP abriga dados meteorológicos diários em forma digital, de séries históricas das várias estações meteorológicas convencionais da rede de estações do INMET com milhões de informações, referentes às medições diárias, de acordo com as normas técnicas internacionais da Organização Meteorológica Mundial.

**Organização**

Instituto Nacional de Meteorologia (INMET)

**Cobertura temporal**

2000 - 2025-12-31

In [21]:
query = """
SELECT
    dados.ano as ano,
    dados.mes as mes,
    dados.data_hora as data_hora,
    dados.bioma as bioma,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.latitude as latitude,
    dados.longitude as longitude,
    dados.satelite as satelite,
    dados.dias_sem_chuva as dias_sem_chuva,
    dados.precipitacao as precipitacao,
    dados.risco_fogo as risco_fogo,
    dados.potencia_radiativa_fogo as potencia_radiativa_fogo
FROM `basedosdados.br_inpe_queimadas.microdados` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano >= 2022 AND ano <= 2025 AND sigla_uf = 'MG'
    """
caminho_completo = config_path.RAW_DATA_DIRECTORY_PATH / "dados_meteorologicos_BDMEP.parquet"
if not caminho_completo.exists():
    df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
    df.to_parquet(caminho_completo, index=False)
    
    
df = pd.read_parquet(caminho_completo)
df

Downloading: 100%|██████████|


,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2025,1,2025-01-18 04:25:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.171250,-45.545000,NOAA-21,2.0,0.00,0.00,0.7
1,2025,1,2025-01-20 17:04:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.209980,-45.376830,NOAA-20,3.0,0.45,0.01,5.4
2,2025,1,2025-01-21 00:26:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.216299,-45.369301,METOP-B,3.0,0.00,0.04,NaN
3,2025,1,2025-01-21 03:27:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.207200,-45.380340,NOAA-21,3.0,0.00,0.03,1.9
4,2025,1,2025-01-21 03:51:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.206250,-45.380340,NPP-375D,3.0,0.00,0.04,1.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405027,2023,8,2023-08-31 22:26:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,114.4
405028,2023,8,2023-08-31 22:16:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,108.6
405029,2023,8,2023-08-31 22:06:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,101.3
405030,2023,8,2023-08-31 22:46:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,94.1


In [22]:
df['data_hora'] = pd.to_datetime(df['data_hora'], format='%Y/%m/%d %H:%M:%S')
    
# Extrai colunas separadas
df['Data'] = df['data_hora'].dt.date       # Só a data (2025-01-01)
df['Hora'] = df['data_hora'].dt.time       # Só a hora (03:49:00)
df['Ano'] = df['data_hora'].dt.year        # Ano
df['Mes'] = df['data_hora'].dt.month       # Mês numérico
df['Dia'] = df['data_hora'].dt.day         # Dia

In [23]:
print(df[['satelite', 'dias_sem_chuva', 'precipitacao', 'risco_fogo', 'potencia_radiativa_fogo']].isna().sum())

satelite                    7790
dias_sem_chuva             15382
precipitacao               15382
risco_fogo                 15382
potencia_radiativa_fogo    19815
dtype: int64


In [24]:
# Substituir strings vazias por NaN
df['satelite'].replace('', pd.NA, inplace=True)

# Preencher colunas categóricas
df['satelite'] = df['satelite'].fillna(df['satelite'].mode()[0])

# Preencher numéricas
num_cols = ['dias_sem_chuva', 'precipitacao', 'risco_fogo', 'potencia_radiativa_fogo']

# Interpolação linear (boa para séries temporais)
df[num_cols] = df[num_cols].interpolate(method='linear')

# Para garantir que não ficou nenhum NaN
df[num_cols] = df[num_cols].fillna(0)

df = df.drop_duplicates()

df['dias_sem_chuva'] = df['dias_sem_chuva'].astype(float)
df['precipitacao'] = df['precipitacao'].astype(float)
df['risco_fogo'] = df['risco_fogo'].astype(float)
df['potencia_radiativa_fogo'] = df['potencia_radiativa_fogo'].astype(float)

In [25]:
df

,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo,Data,Hora,Ano,Mes,Dia
0,2025,1,2025-01-18 04:25:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.171250,-45.545000,NOAA-21,2.0,0.00,0.00,0.70,2025-01-18,04:25:00,2025,1,18
1,2025,1,2025-01-20 17:04:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.209980,-45.376830,NOAA-20,3.0,0.45,0.01,5.40,2025-01-20,17:04:00,2025,1,20
2,2025,1,2025-01-21 00:26:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.216299,-45.369301,METOP-B,3.0,0.00,0.04,3.65,2025-01-21,00:26:00,2025,1,21
3,2025,1,2025-01-21 03:27:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.207200,-45.380340,NOAA-21,3.0,0.00,0.03,1.90,2025-01-21,03:27:00,2025,1,21
4,2025,1,2025-01-21 03:51:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.206250,-45.380340,NPP-375D,3.0,0.00,0.04,1.50,2025-01-21,03:51:00,2025,1,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405027,2023,8,2023-08-31 22:26:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,114.40,2023-08-31,22:26:33,2023,8,31
405028,2023,8,2023-08-31 22:16:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,108.60,2023-08-31,22:16:33,2023,8,31
405029,2023,8,2023-08-31 22:06:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,101.30,2023-08-31,22:06:33,2023,8,31
405030,2023,8,2023-08-31 22:46:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,94.10,2023-08-31,22:46:33,2023,8,31


In [26]:
df.to_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "dados_meteorologicos_BDMEP_processado.parquet", index=False)

In [27]:
del df

### [INMET (Instituto Nacional de Meteorologia)](https://bdmep.inmet.gov.br/)

O Instituto Nacional de Meteorologia (INMET) é o órgão oficial federal do Brasil, vinculado ao Ministério da Agricultura e Pecuária, responsável pelo monitoramento, previsão do tempo e emissão de alertas meteorológicos. Criado em 1909, atua com uma rede de estações automáticas e convencionais para fornecer dados essenciais ao setor agropecuário e à população.

In [28]:
url = "https://portal.inmet.gov.br/uploads/dadoshistoricos/2025.zip"

dir_caminho_inmet = config_path.RAW_DATA_DIRECTORY_PATH / "dataset_inmet"
dir_caminho_inmet_2025 = dir_caminho_inmet / "2025"
dir_caminho_inmet.mkdir(exist_ok=True, parents=True)

zip_file_path = utils.download_file(url, "inmet_2025.zip", dir_caminho_inmet)
utils.unzip_and_clean(zip_file_path, 'all', dir_caminho_inmet)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\dataset_inmet/inmet_2025.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\dataset_inmet
Arquivos mantidos: [WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_S_RS_B827_BAGE - CENTRO_01-08-2025_A_20-10-2025.CSV'), WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025.CSV'), WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025.CSV'), WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2025_A_31-12-2025.CSV'), WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2025_A_31-12-2025.CSV'), WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/ra

[WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_S_RS_B827_BAGE - CENTRO_01-08-2025_A_20-10-2025.CSV'),
 WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025.CSV'),
 WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025.CSV'),
 WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2025_A_31-12-2025.CSV'),
 WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2025_A_31-12-2025.CSV'),
 WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2025_A_31-12-2025.CSV'),
 WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/dataset_inmet/2025/INMET_CO_GO_A002_GOIANIA_01-01-2025_A_

In [29]:
arquivos = list(dir_caminho_inmet_2025.glob("*.CSV"))
lista_cadastro = []

print(f"Iniciando extração de metadados de {len(arquivos)} arquivos...")

for arquivo in arquivos:
    try:
        # Lemos apenas as primeiras 8 linhas para não carregar o arquivo inteiro (ganha velocidade)
        with open(arquivo, 'r', encoding='latin1') as f:
            # Extrai as linhas, limpa espaços e separa pelo ';'
            linhas = [next(f).strip().split(';') for _ in range(7)]
            
            # Criamos um dicionário temporário para organizar os dados daquela estação
            # re.sub remove o ":" dos nomes (ex: "REGIAO:" vira "REGIAO")
            dados_estacao = {
                re.sub(r':', '', l[0]).strip(): l[1].strip() 
                for l in linhas if len(l) > 1
            }
            
            # Adicionamos o nome do arquivo original para conferência
            dados_estacao['ARQUIVO_ORIGEM'] = arquivo.name
            
            lista_cadastro.append(dados_estacao)
            
    except Exception as e:
        print(f"Erro ao ler cabeçalho de {arquivo.name}: {e}")

# 2. Gerar o DataFrame de Cadastro
df_cadastro = pd.DataFrame(lista_cadastro)

# 3. Limpeza Final
# Converter Lat/Long para formato numérico (trocando vírgula por ponto)
for col in ['LATITUDE', 'LONGITUDE', 'ALTITUDE']:
    if col in df_cadastro.columns:
        df_cadastro[col] = df_cadastro[col].str.replace(',', '.').astype(float)

# 4. Salvar o "Mapa das Estações"
caminho_salvamento = config_path.PROCESSED_DATA_DIRECTORY_PATH / "cadastro_estacoes_inmet_2025.csv"
df_cadastro.to_csv(caminho_salvamento, index=False, sep=';', encoding='utf-8-sig')

print(f"\nSucesso! Arquivo de cadastro gerado em: {caminho_salvamento}")
print(df_cadastro.head())

Iniciando extração de metadados de 594 arquivos...

Sucesso! Arquivo de cadastro gerado em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\processed\cadastro_estacoes_inmet_2025.csv
  REGIAO  UF             ESTACAO CODIGO (WMO)   LATITUDE  LONGITUDE  ALTITUDE  \
0     CO  DF            BRASILIA         A001 -15.789444 -47.925833   1160.96   
1     CO  DF          BRAZLANDIA         A042 -15.599722 -48.131111   1143.00   
2     CO  DF     AGUAS EMENDADAS         A045 -15.596389 -47.625833   1030.36   
3     CO  DF   GAMA (PONTE ALTA)         A046 -15.935278 -48.137500    990.00   
4     CO  DF  PARANOA (COOPA-DF)         A047 -16.012222 -47.557500   1043.00   

                                      ARQUIVO_ORIGEM  
0  INMET_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2...  
1  INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12...  
2  INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2025_A_...  
3  INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2025_...  
4  INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2025...  


In [30]:
df_cadastro

,REGIAO,UF,ESTACAO,CODIGO (WMO),LATITUDE,LONGITUDE,ALTITUDE,ARQUIVO_ORIGEM
0,CO,DF,BRASILIA,A001,-15.789444,-47.925833,1160.96,INMET_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2...
1,CO,DF,BRAZLANDIA,A042,-15.599722,-48.131111,1143.00,INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12...
2,CO,DF,AGUAS EMENDADAS,A045,-15.596389,-47.625833,1030.36,INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2025_A_...
3,CO,DF,GAMA (PONTE ALTA),A046,-15.935278,-48.137500,990.00,INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2025_...
4,CO,DF,PARANOA (COOPA-DF),A047,-16.012222,-47.557500,1043.00,INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2025...
...,...,...,...,...,...,...,...,...
589,S,SC,ARARANGUA,A867,-28.931353,-49.497920,2.00,INMET_S_SC_A867_ARARANGUA_01-01-2025_A_31-12-2...
590,S,SC,ITAJAI,A868,-26.950833,-48.761944,9.76,INMET_S_SC_A868_ITAJAI_01-01-2025_A_31-12-2025...
591,S,SC,RANCHO QUEIMADO,A870,-27.678611,-49.041944,881.00,INMET_S_SC_A870_RANCHO QUEIMADO_01-01-2025_A_3...
592,S,SC,CHAPECO,A895,-27.085311,-52.635711,679.00,INMET_S_SC_A895_CHAPECO_01-01-2025_A_31-12-202...


In [31]:
arquivos = list(dir_caminho_inmet_2025.glob("*.CSV"))
lista_df = []

for arquivo in arquivos:
    try:
        # 1. Ler metadados com segurança
        with open(arquivo, encoding='latin1') as f:
            meta = [next(f).strip().split(';') for _ in range(8)]
            # Remove o ':' e limpa espaços dos nomes das chaves
            meta_dict = {re.sub(r'[:]', '', linha[0]).strip(): linha[1].strip() for linha in meta if len(linha) > 1}

        # 2. Ler os dados
        df_meta = pd.read_csv(
            arquivo,
            sep=';',
            nrows=8,
            header=None,
            index_col=0,
            encoding='latin1',
            decimal=','
            
        )        
        
        df_linha = df_meta.T
        df_linha.columns = [col.replace(':', '').strip() for col in df_linha.columns]
        df_linha.reset_index(drop=True, inplace=True)
        
        df = pd.read_csv(
            arquivo,
            sep=';',
            skiprows=8,
            decimal=',',
            encoding='latin1',
            low_memory=False,
            na_values=['', '-9999', '9999,9']  # adiciona os valores que representam nulos
        )
        
        df.drop(columns=['Unnamed: 19'], inplace=True, axis=1)

        # 3. NORMALIZAÇÃO BRAVA DAS COLUNAS (O pulo do gato)
        # Em vez de confiar em '', limpamos tudo que não é letra/número
        df.columns = [re.sub(r'[^a-zA-Z0-9]', '', col).upper() for col in df.columns]

        # Mapeamento por palavras-chave (muito mais seguro)
        mapeamento = {
            'PRECIPITAOTOTALHORRIOMM' : 'precip_total_mm',
            'PRESSAOATMOSFERICAAONIVELDAESTACAOHORARIAMB' : 'pressao_atms_nivel_estacao_mB',
            'PRESSOATMOSFERICAMAXNAHORAANTAUTMB' : 'pressao_atms_maximo_na_hora_mB',
            'PRESSOATMOSFERICAMINNAHORAANTAUTMB' : 'pressao_atms_minimo_na_hora_mB',
            'RADIACAOGLOBALKJM' : 'radiacao_global_kj_m',
            'TEMPERATURADOARBULBOSECOHORARIAC' : 'temp_ar_bulbo_seco_C',
            'TEMPERATURADOPONTODEORVALHOC' : 'tempo_ponto_orvarlho_C',
            'TEMPERATURAMXIMANAHORAANTAUTC' : 'temp_max_hora_C',
            'TEMPERATURAMNIMANAHORAANTAUTC' : 'temp_min_hora_C',
            'TEMPERATURAORVALHOMAXNAHORAANTAUTC' : 'temp_orvalho_max_C',
            'TEMPERATURAORVALHOMINNAHORAANTAUTC' : 'temp_orvalho_min_C',
            'UMIDADERELMAXNAHORAANTAUT' : 'umidade_rel_max',
            'UMIDADERELMINNAHORAANTAUT' : 'umidade_rel_min',
            'UMIDADERELATIVADOARHORARIA' : 'umidade_relativa_ar',
            'VENTODIREOHORARIAGRGR' : 'vento_direcao_horaria',
            'VENTORAJADAMAXIMAMS' : 'vento_rajada_max',
            'VENTOVELOCIDADEHORARIAMS' : 'vento_velocidade_horaria',
            'HORAUTC' : 'hora_utc',
            'DATA' : 'data',
        }
        cols_to_interp = [
            'precip_total_mm',
            'pressao_atms_nivel_estacao_mB',
            'pressao_atms_maximo_na_hora_mB',
            'pressao_atms_minimo_na_hora_mB',
            'radiacao_global_kj_m',
            'temp_ar_bulbo_seco_C',
            'tempo_ponto_orvarlho_C',
            'temp_max_hora_C',
            'temp_min_hora_C',
            'temp_orvalho_max_C',
            'temp_orvalho_min_C',
            'umidade_rel_max',
            'umidade_rel_min',
            'umidade_relativa_ar',
            'vento_direcao_horaria',
            'vento_rajada_max',
            'vento_velocidade_horaria'
        ]
        df.rename(columns=mapeamento, inplace=True)
        
        for col in cols_to_interp:
            # Preenche extremidades para evitar erro
            df[col] = df[col].ffill().bfill()
            # Interpolação polinomial
            df[col] = df[col].interpolate(method='polynomial', order=2)
            
                
        # 4. Criar coluna datetime (Tratando o formato 0000 UTC ou 00:00)
        # Removemos o ' UTC' para o pandas não se confundir
        df['hora'] = df['hora_utc'].str.replace(' UTC', '', regex=False).str.replace(':', '', regex=False)
        df['data_hora'] = pd.to_datetime(df['data'] + ' ' + df['hora'], 
                                        format='%Y/%m/%d %H%M', errors='coerce')

        df['REGIAO'] = df_linha['REGIAO'].iloc[0]
        df['UF'] = df_linha['UF'].iloc[0]
        df['ESTACAO'] = df_linha['ESTACAO'].iloc[0]
        df['CODIGO (WMO)'] = df_linha['CODIGO (WMO)'].iloc[0]
        df['LATITUDE'] = df_linha['LATITUDE'].iloc[0]
        df['LONGITUDE'] = df_linha['LONGITUDE'].iloc[0]
        df['ALTITUDE'] = df_linha['ALTITUDE'].iloc[0]
        df['DATA DE FUNDACAO'] = df_linha['DATA DE FUNDACAO'].iloc[0]
            
        lista_df.append(df)

    except Exception as e:
        print(f"Erro fatal no arquivo {arquivo.name}: {e}")

# Concatenar e Salvar
if lista_df:
    df_inmet_final = pd.concat(lista_df, ignore_index=True)
    df_inmet_final.sort_values('data_hora', inplace=True)
    df_inmet_final.reset_index(inplace=True, drop=True)
    
    for col in cols_to_interp:
        df_inmet_final[col] = df_inmet_final[col].ffill().bfill()
        df_inmet_final[col] = df_inmet_final[col].interpolate(method='polynomial', order=2)
    
    # Salvar em Parquet (MUITO melhor para 1000 arquivos)
    df_inmet_final.to_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "df_inmet_2025.parquet")
    print(f"Processamento concluído: {len(df_inmet_final)} linhas unificadas.")

Processamento concluído: 5020248 linhas unificadas.


In [32]:
df = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "df_inmet_2025.parquet")

df

,data,hora_utc,precip_total_mm,pressao_atms_nivel_estacao_mB,pressao_atms_maximo_na_hora_mB,pressao_atms_minimo_na_hora_mB,radiacao_global_kj_m,temp_ar_bulbo_seco_C,tempo_ponto_orvarlho_C,temp_max_hora_C,...,hora,data_hora,REGIAO,UF,ESTACAO,CODIGO (WMO),LATITUDE,LONGITUDE,ALTITUDE,DATA DE FUNDACAO
0,2025/01/01,0000 UTC,0.0,886.1,886.1,885.5,1.7,20.8,19.5,20.9,...,0000,2025-01-01 00:00:00,CO,DF,BRASILIA,A001,"-15,78944444","-47,92583332","1160,96",07/05/00
1,2025/01/01,0000 UTC,0.0,919.5,919.6,919.3,39.7,20.5,19.9,20.7,...,0000,2025-01-01 00:00:00,SE,MG,RIO PARDO DE MINAS,A551,"-15,72305554","-42,43583333","850,06",17/11/07
2,2025/01/01,0000 UTC,0.0,982.6,982.6,982.2,12.6,23.5,22.0,24.2,...,0000,2025-01-01 00:00:00,SE,MG,ITAOBIM,A550,"-16,57555554","-41,48555555","271,63",05/09/07
3,2025/01/01,0000 UTC,0.0,929.4,929.4,929.2,35.3,20.6,19.8,20.9,...,0000,2025-01-01 00:00:00,SE,MG,AGUAS VERMELHAS,A549,"-15,75166666","-41,45777777","754,07",09/09/07
4,2025/01/01,0000 UTC,0.0,918.4,918.4,917.8,1003.0,24.0,20.6,24.1,...,0000,2025-01-01 00:00:00,SE,MG,CHAPADA GAUCHA,A548,"-15,30027777","-45,61749999","873,2",22/06/07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5020243,2025/12/31,2300 UTC,0.0,1004.8,1004.8,1004.1,2.1,27.2,23.3,28.1,...,2300,2025-12-31 23:00:00,N,AM,EIRUNEPE,A109,"-6,65027777","-69,8686111","121,54",31/08/12
5020244,2025/12/31,2300 UTC,0.0,996.8,996.8,996.0,40.7,27.3,23.5,28.6,...,2300,2025-12-31 23:00:00,N,AM,BOCA DO ACRE,A110,"-8,77666666","-67,33249999","111,98",15/07/08
5020245,2025/12/31,2300 UTC,0.0,1001.6,1003.3,1001.6,2850.5,30.4,22.8,31.0,...,2300,2025-12-31 23:00:00,N,AM,LABREA,A111,"-7,26055555","-64,7886111","61,91",27/07/08
5020246,2025/12/31,2300 UTC,0.0,1001.6,1003.3,1001.6,2850.5,30.4,22.8,31.0,...,2300,2025-12-31 23:00:00,N,AM,APUI,A113,"-7,20555555","-59,8886111","156,97",19/07/08


In [33]:
print(df.isna().sum())

data                              0
hora_utc                          0
precip_total_mm                   0
pressao_atms_nivel_estacao_mB     0
pressao_atms_maximo_na_hora_mB    0
pressao_atms_minimo_na_hora_mB    0
radiacao_global_kj_m              0
temp_ar_bulbo_seco_C              0
tempo_ponto_orvarlho_C            0
temp_max_hora_C                   0
temp_min_hora_C                   0
temp_orvalho_max_C                0
temp_orvalho_min_C                0
umidade_rel_max                   0
umidade_rel_min                   0
umidade_relativa_ar               0
vento_direcao_horaria             0
vento_rajada_max                  0
vento_velocidade_horaria          0
hora                              0
data_hora                         0
REGIAO                            0
UF                                0
ESTACAO                           0
CODIGO (WMO)                      0
LATITUDE                          0
LONGITUDE                         0
ALTITUDE                    

### [MapBiomas](https://brasil.mapbiomas.org/)

O MapBiomas é uma rede global e multi-institucional, formada por universidades, ONGs e empresas de tecnologia, que monitora as transformações na cobertura e no uso da terra nos territórios e seus impactos. Em 2025, a rede completou dez anos, fornecendo a mais atualizada e detalhada base de dados espaciais de uso da terra em um país disponível no mundo. Nascido no Brasil, o MapBiomas está atualmente presente em 14 países – toda a América do Sul e Indonésia.

Com base em ciência aberta e colaborativa, a rede alimenta uma plataforma que integra imagens de satélite, aprendizado de máquina e computação em nuvem. Todos os dados, mapas, métodos e códigos são disponibilizados de forma pública e gratuita.

As informações geradas pela rede MapBiomas podem ser utilizadas por tomadores de decisão, formuladores de políticas públicas, pesquisadores das mais diversas áreas, professores e estudantes, organizações da sociedade civil e empresas.

#### [MapBiomas - Solo]()

#### [MapBiomas - Fogo](https://plataforma.brasil.mapbiomas.org/coverage/coverage_lclu)

##### [MonitorFogo - MapBiomas](https://plataforma.monitorfogo.mapbiomas.org/)

In [34]:
url = "https://docs.google.com/spreadsheets/d/1PZGPIrH1LIFj5Ob54yEVFZiRTOOuJaZ8pfKGZtFnobI/export?format=ods&gid=1141237655"

file_path = utils.download_file(url, "monitor-do-fogo-estatisticas.ods", config_path.RAW_DATA_DIRECTORY_PATH)
df = pd.read_excel(file_path, sheet_name='estatisticas')

df.rename(columns={
    'Ano': 'Ano',
    'Bioma': 'Bioma',
    'Estados': 'Estado',
    'Mês': 'Mes_nome',
    'mes': 'Mes_num',
    'Nível 0': 'Categoria_principal',
    'Nível 1': 'Tipo_uso',
    'Nível 1_1': 'Subtipo_uso',
    'Nível 2': 'Subclasse',
    'Nível 3': 'Classe_fina',
    'Nível 4': 'Classe_final',
    'Área queimada (ha)': 'Area_queimada_ha'
}, inplace=True)

df.to_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "monitor-fogo-mapbiomas.csv", 
          index=False)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/monitor-do-fogo-estatisticas.ods


In [35]:
df

,Ano,Bioma,Estado,Mes_nome,Mes_num,Categoria_principal,Tipo_uso,Subtipo_uso,Subclasse,Classe_fina,Classe_final,Area_queimada_ha
0,2019,Amazônia,Acre,Abril,4,Antrópico,Agropecuária,Pastagem,Pastagem,Pastagem,Pastagem,11.043091
1,2019,Amazônia,Acre,Abril,4,Natural,Floresta,Formação Florestal,Formação Florestal,Formação Florestal,Formação Florestal,0.177260
2,2019,Amazônia,Acre,Agosto,8,Antrópico,Agropecuária,Agricultura,Agricultura,Lavoura Temporária,Outras Lavouras Temporárias,46.909020
3,2019,Amazônia,Acre,Agosto,8,Antrópico,Agropecuária,Pastagem,Pastagem,Pastagem,Pastagem,66041.145360
4,2019,Amazônia,Acre,Agosto,8,Natural,Floresta,Formação Florestal,Floresta Alagável,Floresta Alagável,Floresta Alagável,68.431673
...,...,...,...,...,...,...,...,...,...,...,...,...
25075,2024,Pantanal,Mato Grosso do Sul,Março,3,Antrópico,Corpos D´água,Outros,"Rios, Lagos e Oceano","Rios, Lagos e Oceano","Rios, Lagos e Oceano",381.057779
25076,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Floresta,Formação Florestal,Formação Florestal,Formação Florestal,Formação Florestal,433.061518
25077,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Floresta,Formação Savânica,Formação Savânica,Formação Savânica,Formação Savânica,655.863783
25078,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,2370.983449


In [36]:
del df

### [Banco de Dados de Queimadas - BaseDosDados](https://basedosdados.org/dataset/f06f3cdc-b539-409b-b311-1ff8878fb8d9?table=a3696dc2-4dd1-4f7e-9769-6aa16a1556b8)

Ocorrência do Fogo na Vegetação é o tema deste portal, http://queimadas.dgi.inpe.br/queimadas, desenvolvido no INPE, Instituto Nacional de Pesquisas Espaciais. Ele inclui o monitoramento operacional de focos de queimadas e de incêndios florestais detectados por satélites, e o cálculo e previsão do risco de fogo da vegetação.

Os dados para a América do Sul e a Central, África e Europa, são atualizados a cada três horas, todos os dias do ano. O acesso às informações é livre, e no navegador internet usado, deve estar desbloqueada a carga das janelas "pop up" para acesso a várias figuras, tabelas e gráficos.

Usuários que necessitam de focos com maior frequência temporal ou monitoramento em áreas específicas deverão consultar as "Perguntas Frequentes"

As informações neste portal estão divididas em blocos:

- "Situação Atual", é a "Sala de Situação" do Portal, e fornece para os últimos dois dias os resultados relevantes do monitoramento de focos de queima de vegetação feito pelo INPE em imagens satélites. Vários itens nos títulos das figuras e tabelas podem ser selecionados, alterando as apresentações gráficas conforme as preferências espaciais e temporais do usuário. Passando o mouse sobre os títulos e figuras serão indicadas as opções ativas.
- "SIG Focos-Geral", permite visualizar os focos em um Sistema de Informação Geográfica, com opções de períodos, regiões de interesse, satélites, planos de informação (p.ex. desmatamento, hidrografia, estradas), etc., além da exportação dos dados em formatos txt, html, shp e kmz.
- "SIG Focos-Áreas Protegidas", Semelhante ao item anterior, mas dedicado à ocorrência do fogo em Áreas de Preservação, como Parques, Florestas, Reservas Biológicas municipais, estaduais e nacionais, e Terras Indígenas.
- "Situação nas Áreas Protegidas", que contém o último relatório de focos detectados nas áreas de preservação, incluindo links que os mostram já inseridos no SIG do monitoramento.
- "Relatório Atual", contém o último resumo do monitoramento de queimadas em formato pdf, que poderá ser salvo pelo usuário para análise detalhada dos dados. A opção "Receber por E-Mail" leva ao cadastro do usuário, onde se define individualmente o conteúdo dos relatórios e mensagens de alerta que serão recebidos automaticamente por E-Mail.
- "Risco de Fogo", que apresenta o Risco de Fogo da vegetação estimado no presente, e suas previsões futuras (diárias, semanais e mensais), bem como o "fogograma" para qualquer local selecionado com o mouse nos mapas.
- "Meu Cadastro", onde o usuário define e configura individualmente os produtos que deseja receber em seu E-Mail, como alertas de ocorrência de focos em áreas de preservação, boletins pdf diários com tabelas, gráficos e mapas de estados e países de interesse, e mensagens operacionais.
- "Outros Produtos", com links a produtos adicionais gerados por este sistema de Queimadas e Incêndios, destacando-se: mapas mensais de focos em formato .gif com animações; mapas Meteorológicos (precipitação, temperatura, umidade do ar, ventos); detecções dos vários satélites utilizados; estimativas de concentrações e trajetórias dos poluentes emitidos; localização dos satélites utilizados, uma coletânea de dados anteriores e área de download, e; acesso a uma antiga versão da página do monitoramento de focos.
- "Links Relacionados", com indicações de acesso ao Boletim do Ibama de Monitoramento de Queimadas e Incêndios, ao texto publicado mensalmente no Boletim Climanálise, a uma coleção de mais de 500 páginas com matérias, fotos, vídeos e sites diversos sobre queimadas e incêndios florestais, à composição da equipe deste trabalho, e aos agradecimentos.
- "Notícias e Destaques", com informações, notícias e links de destaque na ocorrência do fogo na vegetação, e não restritas apenas à detecção de focos com satélites.
- "Aviso", com informações operacionais indicando novidades, falhas e detalhes relevantes na geração dos produtos do monitoramento do fogo na vegetação.

**Organização**

Instituto Nacional de Pesquisas Espaciais (INPE)

**Cobertura Temporal**

2000-2025

In [37]:
query = """
SELECT
    dados.ano as ano,
    dados.mes as mes,
    dados.data_hora as data_hora,
    dados.bioma as bioma,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.latitude as latitude,
    dados.longitude as longitude,
    dados.satelite as satelite,
    dados.dias_sem_chuva as dias_sem_chuva,
    dados.precipitacao as precipitacao,
    dados.risco_fogo as risco_fogo,
    dados.potencia_radiativa_fogo as potencia_radiativa_fogo
FROM `basedosdados.br_inpe_queimadas.microdados` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano >= 2020 AND ano <= 2025 AND sigla_uf = 'MG'
    """
caminho_completo = config_path.RAW_DATA_DIRECTORY_PATH / "banco_dados_queimadas_raw.parquet"
if not caminho_completo.exists():
    df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
    
    df_uf = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_ufs.csv")
    df.rename(columns={
                'sigla_uf' : 'Sigla_UF',
                'sigla_uf_nome' : 'Nome_UF',
                'id_municipio' : 'ID_Município',
                'id_municipio_nome' : 'Nome_Município'
                },
                inplace=True)
    df = pd.merge(left=df, right=df_uf, how="inner", on=["Nome_UF"])
    
    
    del df_uf
    
    df.drop(columns=['ano', 'mes'], inplace=True)
    
    df['data_hora'] = pd.to_datetime(df['data_hora'], format='%Y/%m/%d %H:%M:%S')
    
    # Extrai colunas separadas
    df['Data'] = df['data_hora'].dt.date       # Só a data (2025-01-01)
    df['Hora'] = df['data_hora'].dt.time       # Só a hora (03:49:00)
    df['Ano'] = df['data_hora'].dt.year        # Ano
    df['Mes'] = df['data_hora'].dt.month       # Mês numérico
    df['Dia'] = df['data_hora'].dt.day         # Dia
    
    df.dropna(axis=0, inplace=True)
    
    df.to_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "banco_dados_queimadas.parquet", index=False)
    
df = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "banco_dados_queimadas.parquet")
df

Downloading: 100%|██████████|


,data_hora,bioma,Sigla_UF,Nome_UF,ID_Município,Nome_Município,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo,ID_UF,Data,Hora,Ano,Mes,Dia
0,2024-11-12 16:58:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.38586,-47.36140,NOAA-20,2.0,12.71,0.00,3.1,31,2024-11-12,16:58:00,2024,11,12
1,2024-11-13 04:08:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.37368,-47.51899,NOAA-20,2.0,12.85,0.00,1.4,31,2024-11-13,04:08:00,2024,11,13
2,2024-11-13 04:08:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.37182,-47.51684,NOAA-20,2.0,12.80,0.00,3.0,31,2024-11-13,04:08:00,2024,11,13
3,2024-11-20 17:24:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.35582,-47.42835,NPP-375,1.0,4.03,0.02,2.8,31,2024-11-20,17:24:00,2024,11,20
4,2024-11-20 17:24:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.35275,-47.42649,NPP-375,1.0,4.00,0.02,4.2,31,2024-11-20,17:24:00,2024,11,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377828,2024-10-30 17:39:00,Mata Atlântica,MG,Minas Gerais,3169406,Três Pontas,-21.36878,-45.39243,AQUA_M-T,0.0,0.30,0.03,18.8,31,2024-10-30,17:39:00,2024,10,30
377829,2024-10-30 17:41:00,Cerrado,MG,Minas Gerais,3133402,Itapagipe,-19.74592,-49.45157,AQUA_M-T,0.0,0.00,0.63,8.4,31,2024-10-30,17:41:00,2024,10,30
377830,2024-10-30 17:41:00,Cerrado,MG,Minas Gerais,3170206,Uberlândia,-19.15206,-47.93509,NOAA-20,8.0,1.41,0.01,13.6,31,2024-10-30,17:41:00,2024,10,30
377831,2024-10-30 17:41:00,Cerrado,MG,Minas Gerais,3171006,Vazante,-17.82475,-46.73360,AQUA_M-T,8.0,5.00,0.00,11.0,31,2024-10-30,17:41:00,2024,10,30


In [38]:
df.columns

Index(['data_hora', 'bioma', 'Sigla_UF', 'Nome_UF', 'ID_Município',
       'Nome_Município', 'latitude', 'longitude', 'satelite', 'dias_sem_chuva',
       'precipitacao', 'risco_fogo', 'potencia_radiativa_fogo', 'ID_UF',
       'Data', 'Hora', 'Ano', 'Mes', 'Dia'],
      dtype='object')

In [39]:
del df